Before starting, please follow the instructions in the 'Get Started' page to set up an AiiDA profile, computer and CASTEP code. This Notebook uses `computer` and `code` as placeholder names for the computer and code respectively so please change the names accordingly.  Ensure the daemon is running and if not, start it with `verdi daemon start` because it will be needed to submit the workflow.

The first step is to import some necessary modules and ensure this Notebook runs in an AiiDA environment:

In [ ]:
%load_ext aiida
%aiida

from ase.build import bulk
from aiida.engine import submit

You can then import the workflow and get its input builder:

In [ ]:
conv = WorkflowFactory("castep_addons.converge")
bld = conv.get_builder()

The inputs for the builder can be seen using TAB autocompletion (simply type `bld.` and press TAB to see the options).
A good place to start is inputting the code as follows:

In [ ]:
bld.calc.code = Code.get_from_string("code@computer")

If you don't have any pseudopotentials installed yet, you can use these commands to install the C19 pseudopotential family and add it to the builder:

In [ ]:
from aiida_castep.data.otfg import upload_otfg_family
upload_otfg_family(["C19"], "C19", "C19 potential library")
bld.pseudos_family = "C19"

The parameters for the .param and .cell files can be entered using a flat format like with `aiida-castep` workflows:

In [ ]:
bld.calc.parameters = {
    "xc_functional": "lda",
    "cut_off_energy": 300,
    "symmetry_generate": True,
}

There are two ways to set the k-point mesh. The first is using the `kpoints_spacing` input port:

In [ ]:
bld.kpoints_spacing = 0.1

The second way is inputting the k-point mesh as an an AiiDA `KpointsData` node:

In [ ]:
KpointsData = DataFactory("core.array.kpoints")
kpoints = KpointsData()
kpoints.set_kpoints_mesh((4, 4, 4))
bld.calc.kpoints = kpoints

For the structure you can simply use `ase.bulk` to create the primitive unit cell and provide it as `StructureData`:

In [ ]:
StructureData = DataFactory("core.structure")
silicon = StructureData(ase=bulk("Si", "diamond", 5.43))
bld.calc.structure = silicon

The final step is to set the computational resources for the calculations. If your computer uses a `direct` scheduler you can use:

In [ ]:
bld.calc_options = {
"max_wallclock_seconds": 3600,
"resources": {"num_machines": 1, "tot_num_mpiprocs": 4},
}

For computers with different schedulers please refer to this __[official AiiDA page.](https://aiida.readthedocs.io/projects/aiida-core/en/latest/topics/schedulers.html#topics-schedulers-job-resources-node)__

Optionally you can make the workflow clean the remote calculation directories:

In [ ]:
bld.clean_workdir = True

Now you can submit this builder to the daemon:

In [ ]:
submit(bld)

To monitor the status of the workflow, you can use `verdi process list` for all active jobs or `verdi process list -a` for all jobs including completed and failed ones.
You can see the log for the workflow using `verdi process report NODE` and all information including input and output nodes using `verdi node show NODE`.
Using `verdi node graph generate NODE` on a workflow node will generate a provenance graph of the workflow.